# Ανάλυση αποτελεσμάτων

Το notebook διαβάζει τα αποθηκευμένα classification reports των μοντέλων
και υπολογίζει τις διαφορετικές εκδοχές του Macro-F1.


In [1]:
!git clone https://github.com/KontosPetros/thesis.git /kaggle/working/thesis
%cd /kaggle/working/thesis

Cloning into '/kaggle/working/thesis'...
remote: Enumerating objects: 386, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 386 (delta 8), reused 0 (delta 0), pack-reused 371 (from 1)
Receiving objects: 100% (386/386), 525.45 MiB | 33.09 MiB/s, done.
Resolving deltas: 100% (135/135), done.
Updating files: 100% (225/225), done.
/kaggle/working/thesis


In [2]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


RESULTS_DIR = Path("results")
MANIFEST_PATH = Path("data/zoolake_clean_split_manifest.csv")

In [3]:
manifest = pd.read_csv(MANIFEST_PATH)

# Μετράμε πόσες εικόνες έχει συνολικά κάθε κλάση.
class_counts = manifest["label"].value_counts()

class_counts_table = class_counts.to_frame(
    name="number_of_images"
)

display(class_counts_table)

,number_of_images
label,
dinobryon,3321
nauplius,1507
maybe_cyano,1364
diaphanosoma,1089
asterionella,1054
uroglena,953
cyclops,866
ceratium,814
rotifers,744


In [4]:
# Όλες οι κλάσεις του dataset.
all_classes = class_counts.index.tolist()

# Κλάσεις με τουλάχιστον 200 εικόνες συνολικά.
classes_ge_200 = class_counts[
    class_counts >= 200
].index.tolist()

# Κατηγορίες που δεν αντιστοιχούν σε συγκεκριμένο οργανισμό.
excluded_classes = [
    "dirt",
    "unknown",
    "unknown_plankton"
]

classes_no_excluded = [
    class_name
    for class_name in all_classes
    if class_name not in excluded_classes
]

classes_ge_200_no_excluded = [
    class_name
    for class_name in classes_ge_200
    if class_name not in excluded_classes
]


print("Όλες οι κλάσεις:", len(all_classes))
print("Χωρίς τις 3 γενικές κατηγορίες:", len(classes_no_excluded))
print("Με τουλάχιστον 200 εικόνες:", len(classes_ge_200))
print(
    "Με τουλάχιστον 200 εικόνες χωρίς τις γενικές κατηγορίες:",
    len(classes_ge_200_no_excluded)
)

Όλες οι κλάσεις: 35
Χωρίς τις 3 γενικές κατηγορίες: 32
Με τουλάχιστον 200 εικόνες: 23
Με τουλάχιστον 200 εικόνες χωρίς τις γενικές κατηγορίες: 22


In [5]:
report_paths = {
    "Custom CNN": RESULTS_DIR
    / "custom_cnn"
    / "custom_cnn_clean_split_lanczos_seed12345_results"
    / "classification_report.csv",

    "Custom CNN + oversampling": RESULTS_DIR
    / "custom_cnn"
    / "custom_cnn_clean_split_lanczos_oversampling_to100_seed12345_results"
    / "classification_report.csv",

    "EfficientNetB2 frozen": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_frozen_backbone"
    / "classification_report.csv",

    "EfficientNetB2 BN-locked": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_bn_locked_results"
    / "classification_report.csv",

    "EfficientNetB2 BN-locked + oversampling": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_bn_locked_oversampling_results"
    / "classification_report.csv",

    "EfficientNetB2 full unfreeze": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_clean_split_full_unfreeze_results"
    / "classification_report.csv",

    "EfficientNetB2 from scratch": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_clean_split_from_scratch_seed12345_results"
    / "classification_report.csv",

    "MobileNetV1 frozen": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_frozen_backbone_results"
    / "classification_report.csv",

    "MobileNetV1 BN-locked": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_clean_split_bn_locked_results"
    / "classification_report.csv",

    "MobileNetV1 BN-locked + oversampling": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_bn_locked_oversampling_results"
    / "classification_report.csv",

    "MobileNetV1 full unfreeze": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_clean_split_full_unfreeze_results"
    / "classification_report.csv",

    "MobileNetV1 from scratch": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_clean_split_from_scratch_seed12345_results"
    / "classification_report.csv",

    "ConvNeXtBase": RESULTS_DIR
    / "convnextbase"
    / "full_finetune_test_classification_report.csv"
}

In [6]:
reports = {}

for model_name, report_path in report_paths.items():
    report = pd.read_csv(
        report_path,
        index_col=0
    )

    # Στο report του ConvNeXt η κλάση είχε την παλιά ορθογραφία.
    report = report.rename(
        index={"kellikottia": "kellicottia"}
    )

    reports[model_name] = report

print("Διαβάστηκαν reports για", len(reports), "μοντέλα.")

Διαβάστηκαν reports για 13 μοντέλα.


In [7]:
rows = []

for model_name, report in reports.items():

    f1_scores = report["f1-score"]

    rows.append({
        "model": model_name,

        "accuracy": report.loc[
            "accuracy",
            "f1-score"
        ],

        "macro_f1_all": f1_scores.loc[
            all_classes
        ].mean(),

        "macro_f1_no_excluded": f1_scores.loc[
            classes_no_excluded
        ].mean(),

        "macro_f1_ge_200": f1_scores.loc[
            classes_ge_200
        ].mean(),

        "macro_f1_ge_200_no_excluded": f1_scores.loc[
            classes_ge_200_no_excluded
        ].mean()
    })


results_table = pd.DataFrame(rows)

results_table = results_table.set_index("model")

# Μετατρέπουμε τα αποτελέσματα σε ποσοστά.
results_table = results_table * 100

display(results_table.round(2))

,accuracy,macro_f1_all,macro_f1_no_excluded,macro_f1_ge_200,macro_f1_ge_200_no_excluded
model,,,,,
Custom CNN,92.79,74.39,78.28,90.74,92.53
Custom CNN + oversampling,93.83,76.80,81.11,92.64,94.02
EfficientNetB2 frozen,92.72,78.29,82.07,91.13,92.68
EfficientNetB2 BN-locked,95.39,87.34,91.27,94.31,95.70
EfficientNetB2 BN-locked + oversampling,95.13,86.33,90.48,93.99,95.48
EfficientNetB2 full unfreeze,81.72,58.26,61.97,76.70,78.74
EfficientNetB2 from scratch,94.20,79.08,83.30,92.19,93.84
MobileNetV1 frozen,93.79,84.59,89.23,92.66,94.35
MobileNetV1 BN-locked,94.83,83.39,87.36,93.93,95.70


In [8]:
frozen_report = reports[
    "MobileNetV1 frozen"
]

bn_locked_report = reports[
    "MobileNetV1 BN-locked"
]


mobile_comparison = pd.DataFrame({
    "frozen_f1": frozen_report.loc[
        all_classes,
        "f1-score"
    ],

    "bn_locked_f1": bn_locked_report.loc[
        all_classes,
        "f1-score"
    ]
})


mobile_comparison["difference"] = (
    mobile_comparison["bn_locked_f1"]
    - mobile_comparison["frozen_f1"]
).round(10)


display(mobile_comparison)

,frozen_f1,bn_locked_f1,difference
dinobryon,0.972656,0.980392,0.007736
nauplius,0.955357,0.973094,0.017737
maybe_cyano,0.956098,0.963325,0.007228
diaphanosoma,0.975155,0.984326,0.009171
asterionella,0.950495,0.970492,0.019997
uroglena,0.996564,0.996564,0.000000
cyclops,0.928302,0.923077,-0.005225
ceratium,0.968750,0.984252,0.015502
rotifers,0.842105,0.848780,0.006675
daphnia,0.931937,0.949495,0.017558


In [9]:
bn_better = (
    mobile_comparison["difference"] > 0
).sum()

same = (
    mobile_comparison["difference"] == 0
).sum()

frozen_better = (
    mobile_comparison["difference"] < 0
).sum()


print("Καλύτερο το BN-locked:", bn_better)
print("Ίδιο F1:", same)
print("Καλύτερο το frozen:", frozen_better)

Καλύτερο το BN-locked: 23
Ίδιο F1: 5
Καλύτερο το frozen: 7


In [10]:
display(
    mobile_comparison.loc[["chaoborus"]]
)

test_images = frozen_report.loc[
    "chaoborus",
    "support"
]

effect = (
    mobile_comparison.loc[
        "chaoborus",
        "frozen_f1"
    ]
    - mobile_comparison.loc[
        "chaoborus",
        "bn_locked_f1"
    ]
) / len(all_classes)


print("Εικόνες chaoborus στο test:", int(test_images))
print(
    "Επίδραση στο Macro-F1:",
    round(effect * 100, 2),
    "ποσοστιαίες μονάδες"
)

,frozen_f1,bn_locked_f1,difference
chaoborus,1.0,0.0,-1.0


Εικόνες chaoborus στο test: 1
Επίδραση στο Macro-F1: 2.86 ποσοστιαίες μονάδες


In [11]:
classes_without_chaoborus = [
    class_name
    for class_name in all_classes
    if class_name != "chaoborus"
]


frozen_without_chaoborus = frozen_report.loc[
    classes_without_chaoborus,
    "f1-score"
].mean()

bn_locked_without_chaoborus = bn_locked_report.loc[
    classes_without_chaoborus,
    "f1-score"
].mean()


print(
    "Frozen χωρίς chaoborus:",
    round(frozen_without_chaoborus * 100, 2),
    "%"
)

print(
    "BN-locked χωρίς chaoborus:",
    round(bn_locked_without_chaoborus * 100, 2),
    "%"
)

Frozen χωρίς chaoborus: 84.14 %
BN-locked χωρίς chaoborus: 85.84 %


In [12]:
frozen_predictions = pd.read_csv(
    RESULTS_DIR
    / "mobilenet"
    / "mobilenet_frozen_backbone_results"
    / "test_predictions.csv"
)

bn_locked_predictions = pd.read_csv(
    RESULTS_DIR
    / "mobilenet"
    / "mobilenet_clean_split_bn_locked_results"
    / "test_predictions.csv"
)


print("Frozen:")
display(
    frozen_predictions[
        frozen_predictions["true_label"] == "chaoborus"
    ][
        ["true_label", "predicted_label", "confidence"]
    ]
)


print("BN-locked:")
display(
    bn_locked_predictions[
        bn_locked_predictions["true_label"] == "chaoborus"
    ][
        ["true_label", "predicted_label", "confidence"]
    ]
)

Frozen:


,true_label,predicted_label,confidence
2633,chaoborus,chaoborus,0.810433


BN-locked:


,true_label,predicted_label,confidence
2633,chaoborus,leptodora,0.89198
